<a href="https://colab.research.google.com/github/honordold/DS1001-LABS-Projects/blob/main/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [10]:
import requests, pandas as pd, time
from datetime import datetime, timezone

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

class FetchFailed(Exception):
    """Every attempt failed for a reason worth retrying (network or server side)."""

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_problem = None
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:
            last_problem = f'network error: {e}'
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                # 400/404 etc. are our bug: fail right away, no retry
                r.raise_for_status()
                raise RuntimeError(f'unexpected status {r.status_code}')
            last_problem = f'HTTP {r.status_code}'
        if attempt < tries:
            time.sleep(2 ** attempt)   # 2s, then 4s
    raise FetchFailed(f'all {tries} attempts failed; last problem: {last_problem}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [11]:
START, END = '2026-09-05', '2026-10-03'
WANTED = ['2026-09-05', '2026-10-03']
used_fallback = False

try:
    raw = get_weather(START, END)
    wx = raw[raw['time'].isin(WANTED)].reset_index(drop=True)
except FetchFailed as e:
    print(f'API call failed, using fallback sample: {e}')
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [12]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [13]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [14]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [15]:
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']
note = ' (NOTE: based on fallback sample data, not real observations)' if used_fallback else ''

if diff == 0:
    print(f"Both dates had the same rainfall in Charlottesville ({wet['precip_mm']:.1f} mm){note}.")
else:
    print(f"{wet['date']:%A, %B} {wet['date'].day}, {wet['date'].year} was the wetter day in "
          f"Charlottesville, with {wet['precip_mm']:.1f} mm of rain versus "
          f"{dry['precip_mm']:.1f} mm on {dry['date']:%A, %B} {dry['date'].day}, "
          f"a difference of {diff:.1f} mm{note}.")

Saturday, October 3, 2026 was the wetter day in Charlottesville, with 14.7 mm of rain versus 2.5 mm on Saturday, September 5, a difference of 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [16]:
pulled_at = datetime.now(timezone.utc).astimezone()
print('Source:      ', BASE)
print('Coordinates: ', '38.03, -78.51 (Charlottesville, VA)')
print('Date range:  ', f'{START} to {END} (kept {", ".join(WANTED)})')
print('Pulled at:   ', pulled_at.strftime('%Y-%m-%d %H:%M %Z'))
print('Data status: ', 'FALLBACK SAMPLE (not real)' if used_fallback else 'real data from Open-Meteo archive')

Source:       https://archive-api.open-meteo.com/v1/archive
Coordinates:  38.03, -78.51 (Charlottesville, VA)
Date range:   2026-09-05 to 2026-10-03 (kept 2026-09-05, 2026-10-03)
Pulled at:    2026-10-11 15:38 UTC
Data status:  real data from Open-Meteo archive


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_If get_weather returned an empty table instead of raising an error, the notebook would keep running. Merging it onto 200 days of sales would keep all 200 sales rows, but every weather value would be blank. The chart of rain against revenue would either be empty or skip the blank points. If the blanks were filled with 0, it would look like it never rained, and the chart would suggest rain has no effect on sales. This would be hard to catch because nothing errors. The code runs, the columns look right, and the table is the right size. The mistake shows up in the conclusion instead of as an error, which is why the function should fail loudly and the data should be checked before it’s used._

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [17]:
CITIES = {
    'Charlottesville': (38.03, -78.51),
    'Orlando': (28.54, -81.38),
}

frames = []
for city, (lat, lon) in CITIES.items():
    try:
        raw = get_weather(START, END, lat=lat, lon=lon)
        df = raw[raw['time'].isin(WANTED)].copy()
        df['source'] = 'open-meteo'
    except FetchFailed as e:
        print(f'{city}: API failed, no fallback sample for this city ({e})')
        continue
    df['city'] = city
    frames.append(df)

both = pd.concat(frames, ignore_index=True)
both = both.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
both['date'] = pd.to_datetime(both['date'])
both = both[['city', 'date', 'temp_max', 'precip_mm', 'source']]

assert len(both) == 4, f'expected 4 rows (2 cities x 2 dates), got {len(both)}'
both

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
